# investalyze — ticker profile

Everything we hold for one ticker, table by table: a metadata summary plus a head/tail
preview of the rows. Set `TICKER` and run all.

The notebook probes every table, so it works for both stocks (prices + dividends/splits +
income/balance/cashflow + companies) and non-equity tickers (market_data). Run from the
repo root (so `ingest.toml` resolves).

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Markdown, display

from investalyze.ingest import config, storage

display(HTML('<style>table.dataframe td {white-space: nowrap;}</style>'))
pd.set_option('display.max_columns', None)

# resolve the repo root (walk up to ingest.toml) so paths work regardless of CWD
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ingest.toml').exists())
cfg = config.load(root / 'ingest.toml')
con = storage.connect(root / cfg.data_root, read_only=True)  # explore only, never mutate
existing = {name for (name,) in con.execute('SHOW TABLES').fetchall()}

TICKER = 'AAPL'    # the one knob — set the ticker to profile
MIN_FILL = 0.5     # fundamentals: only stat columns filled in >= this fraction of rows
                   # (drops NaN-heavy breakdown columns like 'Other Revenue')

In [2]:
def load(table, ticker):
    """All rows for `ticker` in `table` as a DataFrame (empty if the table is absent)."""
    if table not in existing:
        return pd.DataFrame()
    return con.execute(f'SELECT * FROM {table} WHERE Ticker = ?', [ticker]).df()


def head_tail(df, n=5):
    """First `n` + last `n` rows; the whole frame if it has <= 2n rows."""
    if len(df) <= 2 * n:
        return df
    return pd.concat([df.head(n), df.tail(n)])


def ts_meta(df):
    """count/min/max/mean/std per numeric column of a time series (Date span shown separately)."""
    return df.select_dtypes('number').describe().T[['count', 'min', 'max', 'mean', 'std']]


def fund_meta(df):
    """One-line coverage summary for a fundamentals slice."""
    fy = f"{int(df['Fiscal Year'].min())} .. {int(df['Fiscal Year'].max())}"
    periods = ' '.join(f'{p}={n}' for p, n in df['Period'].value_counts().sort_index().items())
    currency = ', '.join(sorted(df['Currency'].dropna().unique()))
    return f'Fiscal Year {fy}  |  {periods}  |  {currency}  |  {len(df)} rows'


def fund_stats(df):
    """count/min/max/mean/std for the well-populated numeric columns of a fundamentals slice.

    Importance heuristic: a column's NaN fraction. Columns filled in fewer than MIN_FILL of
    the rows (e.g. 'Sales & Services Revenue', 'Other Revenue') are dropped; the core line
    items (Revenue, Cost of Revenue, ...) survive. Ordered most-populated first.
    """
    num = df.select_dtypes('number').drop(columns=['SrcId', 'Fiscal Year'], errors='ignore')
    fill = num.notna().mean()
    keep = fill[fill >= MIN_FILL].sort_values(ascending=False).index
    return num[keep].describe().T[['count', 'min', 'max', 'mean', 'std']]


def section(title):
    display(Markdown(f'## {title}'))


def note(text):
    display(Markdown(f'*{text}*'))


def ts_section(table, ticker):
    """A time-series table: date span + per-column stats + head/tail preview."""
    section(table)
    df = load(table, ticker).sort_values('Date')
    if df.empty:
        note('no rows')
        return
    note(f'{df.Date.min()} .. {df.Date.max()}  |  {len(df)} rows')
    display(ts_meta(df))
    display(head_tail(df))


def profile(ticker):
    """Everything we hold for `ticker`: metadata + a head/tail preview, table by table."""
    display(Markdown(f'# {ticker}'))

    # identity card — one row, transposed
    section('companies')
    companies = load('companies', ticker)
    note('no rows') if companies.empty else display(companies.T)

    ts_section('prices', ticker)
    ts_section('dividends', ticker)
    ts_section('splits', ticker)

    # fundamentals — split each statement into as-reported vs restated
    for table in ('income', 'balance', 'cashflow'):
        section(table)
        df = load(table, ticker).sort_values(['Fiscal Year', 'Fiscal Period'])
        if df.empty:
            note('no rows')
            continue
        for is_restated, label in ((False, 'as-reported'), (True, 'restated')):
            display(Markdown(f'### {table} ({label})'))
            part = df[df['IsRestated'] == is_restated]
            if part.empty:
                note('no rows')
                continue
            note(fund_meta(part))
            display(fund_stats(part))
            display(head_tail(part))

    ts_section('market_data', ticker)  # populates for non-equity tickers

In [3]:
profile(TICKER)

# AAPL

## companies

,0
Ticker,AAPL
SrcId,111052
Src,simfin
Market,us
Industry,Computer Hardware
Sector,Technology
Company Name,APPLE INC
IndustryId,101001
ISIN,US0378331005
End of financial year (month),9


## prices

*1980-12-12 00:00:00 .. 2026-06-18 00:00:00  |  11471 rows*

...

*Fiscal Year 2000 .. 2026  |  A=26 Q=104  |  USD  |  130 rows*

,count,min,max,mean,std
Shares (Basic),130.0,14673278000.0,26309612000.0,21835233546.153847,3815688906.030132
Shares (Diluted),130.0,14725873000.0,26541056000.0,22181176730.76923,3986293905.81803
Net Income/Starting Line,130.0,-195000000.0,112010000000.0,15892715384.615385,21808741646.887646
Depreciation & Amortization,130.0,18000000.0,12547000000.0,2309446153.846154,3079895948.764771
Non-Cash Items,130.0,-32441000000.0,12774000000.0,1240223076.923077,4576835109.168516
Change in Working Capital,130.0,-25000000000.0,37924000000.0,549730769.230769,5953208625.813207
Net Cash from Operating Activities,130.0,-125000000.0,122151000000.0,19992115384.615383,26240617476.733021
Change in Fixed Assets & Intangibles,130.0,-13313000000.0,-10000000.0,-2583746153.846154,3213470712.537137
Net Change in Long Term Investment,130.0,-44417000000.0,58093000000.0,-1262761538.461539,12738325059.845671
Net Cash from Financing Activities,130.0,-121983000000.0,6884000000.0,-14734607692.307692,25855029916.075916


,Ticker,SrcId,Src,Market,Period,IsRestated,Currency,Fiscal Year,Fiscal Period,Report Date,Publish Date,Restated Date,Shares (Basic),Shares (Diluted),Net Income/Starting Line,Net Income,Net Income from Discontinued Operations,Other Adjustments,Depreciation & Amortization,Non-Cash Items,Stock-Based Compensation,Deferred Income Taxes,Other Non-Cash Adjustments,Change in Working Capital,Change in Accounts Receivable,Change in Inventories,Change in Accounts Payable,Change in Other,Net Cash from Discontinued Operations (Operating),Net Cash from Operating Activities,Change in Fixed Assets & Intangibles,Disposition of Fixed Assets & Intangibles,Disposition of Fixed Assets,Disposition of Intangible Assets,Acquisition of Fixed Assets & Intangibles,Purchase of Fixed Assets,Acquisition of Intangible Assets,Other Change in Fixed Assets & Intangibles,Net Change in Long Term Investment,Decrease in Long Term Investment,Increase in Long Term Investment,Net Cash from Acquisitions & Divestitures,Net Cash from Divestitures,Cash for Acquisition of Subsidiaries,Cash for Joint Ventures,Net Cash from Other Acquisitions,Other Investing Activities,Net Cash from Discontinued Operations (Investing),Net Cash from Investing Activities,Dividends Paid,Cash from (Repayment of) Debt,"Cash from (Repayment of) Short Term Debt, Net","Cash from (Repayment of) Long Term Debt, Net",Repayments of Long Term Debt,Cash from Long Term Debt,Cash from (Repurchase of) Equity,Increase in Capital Stock,Decrease in Capital Stock,Other Financing Activities,Net Cash from Discontinued Operations (Financing),Net Cash from Financing Activities,Net Cash Before Disc. Operations and FX,Change in Cash from Disc. Operations and Other,Net Cash Before FX,Effect of Foreign Exchange Rates,Net Change in Cash
19,AAPL,111052,simfin,us,A,True,USD,2000,FY,2000-09-30,2000-12-19,2002-12-19,24208660000,24900176000,786000000,<NA>,<NA>,<NA>,84000000,-194000000,<NA>,<NA>,<NA>,192000000,<NA>,<NA>,<NA>,<NA>,<NA>,868000000,-142000000,<NA>,<NA>,<NA>,-142000000,<NA>,<NA>,<NA>,3215000000,3447000000,-232000000,<NA>,<NA>,<NA>,<NA>,<NA>,-4045000000,<NA>,-972000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,-31000000,85000000,-116000000,<NA>,<NA>,-31000000,-135000000,<NA>,-135000000,<NA>,-135000000
206,AAPL,111052,simfin,us,Q,True,USD,2000,Q3,2000-06-30,2000-08-13,2001-08-13,24208660000,24900176000,200000000,<NA>,<NA>,<NA>,25000000,-4000000,<NA>,<NA>,<NA>,13000000,<NA>,<NA>,<NA>,<NA>,<NA>,234000000,-10000000,1000000,<NA>,<NA>,-11000000,<NA>,<NA>,<NA>,680000000,696000000,-16000000,<NA>,<NA>,<NA>,<NA>,<NA>,-1175000000,<NA>,-505000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,-47000000,3000000,-50000000,<NA>,<NA>,-47000000,-318000000,<NA>,-318000000,<NA>,-318000000
241,AAPL,111052,simfin,us,Q,True,USD,2000,Q4,2000-09-30,2000-12-19,2002-12-19,24208660000,24900176000,170000000,<NA>,<NA>,<NA>,18000000,-62000000,<NA>,<NA>,<NA>,74000000,<NA>,<NA>,<NA>,<NA>,<NA>,200000000,-77000000,-11000000,<NA>,<NA>,-66000000,<NA>,<NA>,<NA>,718000000,718000000,0,<NA>,<NA>,<NA>,<NA>,<NA>,-1004000000,<NA>,-363000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,10000000,35000000,-25000000,<NA>,<NA>,10000000,-153000000,<NA>,-153000000,<NA>,-153000000
177,AAPL,111052,simfin,us,A,True,USD,2001,FY,2001-09-30,2001-12-21,2003-12-19,24208660000,24900176000,-25000000,<NA>,<NA>,<NA>,100000000,-101000000,<NA>,<NA>,<NA>,211000000,<NA>,<NA>,<NA>,<NA>,<NA>,185000000,-232000000,<NA>,<NA>,<NA>,-232000000,<NA>,<NA>,<NA>,5151000000,5151000000,<NA>,-19000000,<NA>,<NA>,<NA>,<NA>,-4008000000,<NA>,892000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,42000000,42000000,<NA>,<NA>,<NA>,42000000,1119000000,<NA>,1119000000,<NA>,1119000000
51,AAPL,111052,simfin,us,Q,True,USD,2001,Q1,2000-12-31,2001-02-11,2002-02-11,24208660000,24900176000,-195000000,<NA>,<NA>,<NA>,24000000,-162000000,<NA>,<NA>,<NA>,320000000,<NA>,<NA>,<NA>,<NA>,<NA>,-13000000,-22000000,<NA>,<NA>,<NA>,-22000000,<NA>,<NA>,<NA>,1146000000,1147000000,-1000000,<NA>,<NA>,<NA>,<NA>,<NA>,-568000000,<NA>,556000000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,3000000,3000000

## market_data

*no rows*